# Specify GACC

The code below adds the GACC to the wildfire and airports/airtanker bases.

In [ ]:
# import libraries
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

# === Load GACC shapefile ===
gacc_shapefile_path = '../data/raw_data/gacc_boundaries/National_GACC_Final_20250113.shp'
gacc_gdf = gpd.read_file(gacc_shapefile_path)
gacc_gdf = gacc_gdf.rename(columns={'GACCName': 'gacc'})

# === Function to add GACC to a dataset based on lat/lon columns ===
def add_gacc_to_dataframe(df, lon_col, lat_col):
    # Ensure coordinates are numeric
    df[lon_col] = pd.to_numeric(df[lon_col], errors='coerce')
    df[lat_col] = pd.to_numeric(df[lat_col], errors='coerce')
    df = df.dropna(subset=[lon_col, lat_col])

    # Create GeoDataFrame
    geometry = [Point(xy) for xy in zip(df[lon_col], df[lat_col])]
    gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

    # Reproject to GACC CRS if different
    if gdf.crs != gacc_gdf.crs:
        gdf = gdf.to_crs(gacc_gdf.crs)

    # Spatial join with GACC polygons
    gdf_with_gacc = gpd.sjoin(
        gdf,
        gacc_gdf[['geometry', 'gacc']],
        how='left',
        predicate='within',
        lsuffix='',
        rsuffix='_gacc'
    )
    # Drop geometry for output CSV
    return gdf_with_gacc.drop(columns='geometry')

# === Process Wildfire Weather Dataset ===
wildfire_weather_path = '../data/processed_data/Wildfire_Weather_2020_2024.csv'
wildfire_df = pd.read_csv(wildfire_weather_path)

wildfire_with_gacc = add_gacc_to_dataframe(wildfire_df, lon_col='longitude', lat_col='latitude')

wildfire_output_path = '../data/processed_data/Wildfire_Weather_2020_2024_with_gacc.csv'
wildfire_with_gacc.to_csv(wildfire_output_path, index=False)

# Summary for wildfire
wildfire_gacc_regions = wildfire_with_gacc['gacc'].dropna().unique()
print(f"Wildfire weather records tagged with {len(wildfire_gacc_regions)} unique GACC regions.")
print("Regions:", wildfire_gacc_regions)
print(f"Saved wildfire data with GACC to: {wildfire_output_path}")

# === Process airports_processed.csv ===
airports_processed_path = '../data/processed_data/airports_processed.csv'
airports_processed_df = pd.read_csv(airports_processed_path)

airports_processed_with_gacc = add_gacc_to_dataframe(airports_processed_df, lon_col='longitude_deg', lat_col='latitude_deg')

airports_processed_output = '../data/processed_data/airports_processed.csv'
airports_processed_with_gacc.to_csv(airports_processed_output, index=False)

# Summary for airports_processed
airports_processed_gacc_regions = airports_processed_with_gacc['gacc'].dropna().unique()
print(f"Airports processed records tagged with {len(airports_processed_gacc_regions)} unique GACC regions.")
print("Regions:", airports_processed_gacc_regions)
print(f"Saved airports_processed data with GACC to: {airports_processed_output}")

# === Process airports_runways_joined.csv ===
airports_runways_path = '../data/processed_data/airports_runways_joined.csv'
airports_runways_df = pd.read_csv(airports_runways_path)

airports_runways_with_gacc = add_gacc_to_dataframe(airports_runways_df, lon_col='longitude_deg', lat_col='latitude_deg')

airports_runways_output = '../data/processed_data/airports_runways_joined.csv'
airports_runways_with_gacc.to_csv(airports_runways_output, index=False)

# Summary for airports_runways_joined
airports_runways_gacc_regions = airports_runways_with_gacc['gacc'].dropna().unique()
print(f"Airports runways joined records tagged with {len(airports_runways_gacc_regions)} unique GACC regions.")
print("Regions:", airports_runways_gacc_regions)
print(f"Saved airports_runways_joined data with GACC to: {airports_runways_output}")


Wildfire weather records tagged with 10 unique GACC regions.
Regions: ['Southern Area Coordination Center' 'Southwest Area Coordination Center'
 'Rocky Mountain Area Coordination Center'
 'Eastern Area Coordination Center'
 'Northern California Geographic Area Coordination Center'
 'Northwest Interagency Coordination Center'
 'Northern Rockies Coordination Center' 'Great Basin Coordination Center'
 'Southern California Coordination Center'
 'Alaska Interagency Coordination Center']
Saved wildfire data with GACC to: ../data/processed_data/Wildfire_Weather_2020_2024_with_gacc.csv
Airports processed records tagged with 10 unique GACC regions.
Regions: ['Southern Area Coordination Center'
 'Rocky Mountain Area Coordination Center'
 'Eastern Area Coordination Center' 'Great Basin Coordination Center'
 'Southwest Area Coordination Center'
 'Northwest Interagency Coordination Center'
 'Northern Rockies Coordination Center'
 'Northern California Geographic Area Coordination Center'
 'Southern 